# Data Preparation
Now that the training and test data has been created, we can preprocess them. The reason we preprocess them separately is to avoid any data leakage that may be casued by numerical operations like filling out missing values with the mean.

Note that we assume the data has already been encoded beforehand. The reason for this is because encoding does not modify any information, it just changes its representation.

In [6]:
import numpy as np
import pandas as pd

trainSetName = "train_set.csv"
testSetName = "test_set.csv"

trainSetRaw = pd.read_csv(trainSetName)
testSetRaw = pd.read_csv(testSetName)

In [7]:
#Array<Array<String, Dictionary>>
ordinalEncodedFeatures = [
    ["quality", {"none": 0, "poor": 1, "ok": 2, "great": 3}],
    ["condition", {"ok": 1, "wow": 2, "wowie": 3}]
    ]

#Array<String>
oneHotEncodedFeatures = ["a", "colorness"]

#Array<String>
explainedMissingFeatures = []

#Array<String>
unexplainedMissingFeatures = [] 

#Encodes
def encodeData(df : pd.DataFrame, ordinalEncodedFeatures, oneHotEncodedFeatures):
    encoded = df.copy()

    #Convert feature to ordinal using the provided map
    for ord in ordinalEncodedFeatures:
        encoded[ord[0]] = encoded.map(ord[1])

    #One hot encode feature and remove the original column
    for hot in oneHotEncodedFeatures:
        hotDf = pd.get_dummies(encoded, columns=hot) #Get encoding
        encoded.drop(hot, axis=1) #Remove original column
        encoded = pd.concat([encoded, hotDf], axis='columns') #concat columns
        

    return encoded

#All missing values with no explanation are replaced with the mean of that feature
#Otherwise they are set to 0
#E.g. 
#missing pool or missing basement => fill out with 0ft^2 pool/basement
#missing 1st floor surface area => fill out with mean 
def prepData(df : pd.DataFrame, explainedMissing, unexplainedMissing):
    dfPrepped = df.copy()

    #Fill out explained missing
    for f in explainedMissing:
        missing = df[dfPrepped[f].isna()].index.to_list()
        for m in missing:
            dfPrepped[f][m] = 0

    #Fill out unexplained missing
    for f in unexplainedMissing:
        meanVal = df[f].mean()
        missing = df[dfPrepped[f].isna()].index.to_list()
        for m in missing:
            dfPrepped[f][m] = meanVal

    return dfPrepped


trainSetEncoded = encodeData(trainSetRaw, ordinalEncodedFeatures, oneHotEncodedFeatures)
trainSetPrepped = prepData(trainSetEncoded, explainedMissingFeatures, unexplainedMissingFeatures)

testSetEncoded = encodeData(testSetRaw, ordinalEncodedFeatures, oneHotEncodedFeatures)
testSetPrepped = prepData(testSetEncoded, explainedMissingFeatures, unexplainedMissingFeatures)


TypeError: the first argument must be callable

In [ ]:
trainSetPrepped.to_csv("train_set_prepped.csv")
testSetPrepped.to_csv("test_set_prepped.csv")